In [4]:
import pandas as pd
import datetime as dt

# 1. Load the dataset
df = pd.read_csv('online_retail.csv')

# 2. Step 1: Drop missing Customer IDs
clean_df = df.dropna(subset=['Customer ID']).copy()

# 3. Step 2: Filter strictly for positive transactions
clean_df = clean_df[(clean_df['Quantity'] > 0) & (clean_df['Price'] > 0)]

# 4. Step 3: Engineer the Total Amount
clean_df['TotalAmount'] = clean_df['Quantity'] * clean_df['Price']

# 5. Format dates and setup Snapshot
clean_df['InvoiceDate'] = pd.to_datetime(clean_df['InvoiceDate'])
snapshot_date = clean_df['InvoiceDate'].max() + dt.timedelta(days=1)

# 6. Step 4: Extract RFM
rfm = clean_df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'Invoice': 'nunique',                                    # Frequency
    'TotalAmount': 'sum'                                     # Monetary
}).reset_index()

# 7. Rename to standard terms
rfm.rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'TotalAmount': 'Monetary'}, inplace=True)

# Format Customer ID cleanly as a string (avoids scientific notation)
rfm['Customer ID'] = rfm['Customer ID'].astype(int).astype(str)

print(rfm.head())
rfm.to_csv('Retail_RFM_Final.csv', index=False)

  Customer ID  Recency  Frequency  Monetary
0       12346      326         12  77556.46
1       12347        2          8   5633.32
2       12348       75          5   2019.40
3       12349       19          4   4428.69
4       12350      310          1    334.40
